# 03 · Export the chosen model to ONNX (+ INT8)
Turns the PyTorch model into files your laptop can run with **ONNX Runtime** — no PyTorch, much faster on CPU.

1. Export the chosen model with **calibration built in** (outputs calibrated probabilities)
2. Check ONNX matches PyTorch, and that the laptop preprocessing matches training exactly
3. Make an **INT8** version (static quantization) — smaller and usually faster
4. Test-set accuracy + CPU latency for PyTorch / ONNX FP32 / ONNX INT8
5. Save everything to Drive → `driver-monitoring/exports/`, ready to download

**Runtime:** CPU is fine (the benchmark is meant to measure CPU speed). About 10 min.

## 1 · Setup

In [ ]:
GITHUB_USER = "harshithdande28"
REPO_NAME = "driver-monitoring-system"

import os, sys
from google.colab import drive
drive.mount("/content/drive")
REPO_DIR = f"/content/{REPO_NAME}"
if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull
else:
    !git clone https://github.com/{GITHUB_USER}/{REPO_NAME}.git {REPO_DIR}
%cd {REPO_DIR}
!pip install -q -r requirements.txt onnx onnxruntime onnxscript
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

## 2 · Load the chosen model and its temperature

In [ ]:
import json, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch import nn

from src.config import load_config, class_names
from src.data.prepare import extract_dataset
from src.data.dataset import build_dataloaders
from src.data.transforms import build_transforms
from src.models.factory import build_model
from src.training.checkpoint import load_checkpoint
from src.inference.preprocess import preprocess_rgb
from src.evaluation.metrics import multiclass_metrics, binary_metrics

cfg = load_config()
CLASS_NAMES = class_names(cfg)
CLASS_CODES = list(CLASS_NAMES)
IMG = cfg["train"]["image_size"]
DRIVE_ROOT = Path(cfg["paths"]["drive_root"])
EXPORTS = DRIVE_ROOT / "exports"
EXPORTS.mkdir(parents=True, exist_ok=True)

chosen = json.loads((DRIVE_ROOT / "results" / "chosen_model.json").read_text())
NAME, T = chosen["model"], chosen["temperature"]
print(f"Chosen model: {NAME}, temperature {T:.3f}")

model = build_model(NAME, len(CLASS_CODES), pretrained=False)
model.load_state_dict(load_checkpoint(DRIVE_ROOT / "checkpoints" / NAME / "best.pt")["model"])
model.eval()

class CalibratedModel(nn.Module):
    """Image -> calibrated probabilities: softmax(logits / T)."""
    def __init__(self, net, temperature):
        super().__init__()
        self.net = net
        self.register_buffer("inv_t", torch.tensor(1.0 / temperature))
    def forward(self, x):
        return torch.softmax(self.net(x) * self.inv_t, dim=1)

wrapped = CalibratedModel(model, T).eval()

## 3 · Export to ONNX (FP32)

In [ ]:
FP32 = EXPORTS / "distraction_fp32.onnx"
dummy = torch.randn(1, 3, IMG, IMG)
export_kwargs = dict(input_names=["image"], output_names=["probs"],
                     dynamic_axes={"image": {0: "batch"}, "probs": {0: "batch"}}, opset_version=17)
try:
    torch.onnx.export(wrapped, dummy, str(FP32), dynamo=False, **export_kwargs)
except TypeError:   # older PyTorch without the `dynamo` argument
    torch.onnx.export(wrapped, dummy, str(FP32), **export_kwargs)

import onnx, onnxruntime as ort
onnx.checker.check_model(str(FP32))
print(f"Saved {FP32.name}: {FP32.stat().st_size / 1e6:.1f} MB")

## 4 · Checks: ONNX = PyTorch, and laptop preprocessing = training preprocessing

In [ ]:
LOCAL_DATA = extract_dataset(DRIVE_ROOT / "data" / cfg["data"]["zip_name"], cfg["paths"]["local_data"])
split_df = pd.read_csv(cfg["paths"]["splits_file"])
from PIL import Image

sample = Image.open(LOCAL_DATA / split_df.iloc[0]["path"]).convert("RGB")
x_torch = build_transforms(train=False, image_size=IMG)(sample).unsqueeze(0).numpy()
x_laptop = preprocess_rgb(np.asarray(sample), IMG)
print("Preprocessing max difference:", float(np.abs(x_torch - x_laptop).max()))
assert np.allclose(x_torch, x_laptop, atol=1e-5), "Laptop preprocessing does not match training!"

sess = ort.InferenceSession(str(FP32), providers=["CPUExecutionProvider"])
with torch.no_grad():
    p_torch = wrapped(torch.from_numpy(x_torch)).numpy()
p_onnx = sess.run(None, {"image": x_laptop})[0]
print("ONNX vs PyTorch max probability difference:", float(np.abs(p_torch - p_onnx).max()))
assert np.allclose(p_torch, p_onnx, atol=1e-4)
print("✅ Both checks passed")

## 5 · INT8 static quantization
Weights and activations are stored as 8-bit integers. ~200 **training** images are used to measure
activation ranges (never test images). Accuracy is checked below — if INT8 loses more than 1 point,
the FP32 model is used for the demo instead.

In [ ]:
from onnxruntime.quantization import (CalibrationDataReader, QuantFormat, QuantType,
                                      quantize_static)
from onnxruntime.quantization.shape_inference import quant_pre_process

PRE = EXPORTS / "distraction_fp32_pre.onnx"
INT8 = EXPORTS / "distraction_int8.onnx"
quant_pre_process(str(FP32), str(PRE))

calib_paths = (split_df[split_df["split"] == "train"]
               .groupby("classname", group_keys=False)
               .apply(lambda g: g.sample(20, random_state=cfg["seed"]))["path"].tolist())

class Reader(CalibrationDataReader):
    def __init__(self, paths):
        self.it = iter(paths)
    def get_next(self):
        p = next(self.it, None)
        if p is None:
            return None
        return {"image": preprocess_rgb(np.asarray(Image.open(LOCAL_DATA / p).convert("RGB")), IMG)}

quantize_static(str(PRE), str(INT8), Reader(calib_paths), quant_format=QuantFormat.QDQ,
                per_channel=True, activation_type=QuantType.QUInt8, weight_type=QuantType.QInt8)
PRE.unlink()
print(f"FP32 {FP32.stat().st_size / 1e6:.1f} MB  ->  INT8 {INT8.stat().st_size / 1e6:.1f} MB")

## 6 · Test-set accuracy (unseen drivers) for PyTorch, ONNX FP32, ONNX INT8

In [ ]:
loaders = build_dataloaders(split_df, LOCAL_DATA, CLASS_CODES, image_size=IMG, batch_size=64,
                            num_workers=2, seed=cfg["seed"])
test_x, test_y = [], []
for xb, yb in loaders["test"]:
    test_x.append(xb.numpy()); test_y.append(yb.numpy())
test_y = np.concatenate(test_y)

def predict_torch():
    with torch.no_grad():
        return np.concatenate([wrapped(torch.from_numpy(xb)).numpy() for xb in test_x])

def predict_onnx(path):
    s = ort.InferenceSession(str(path), providers=["CPUExecutionProvider"])
    return np.concatenate([s.run(None, {"image": xb})[0] for xb in test_x])

probs = {"pytorch_fp32": predict_torch(), "onnx_fp32": predict_onnx(FP32), "onnx_int8": predict_onnx(INT8)}
acc_rows = {k: {**multiclass_metrics(v, test_y), **binary_metrics(v, test_y)} for k, v in probs.items()}
acc_table = pd.DataFrame(acc_rows).T[["accuracy", "macro_f1", "ece", "bin_recall", "false_alarm_rate"]].round(4)
acc_table

## 7 · CPU latency (one image at a time, like the live demo)

In [ ]:
def bench(fn, x, warmup=10, runs=100):
    for _ in range(warmup):
        fn(x)
    times = []
    for _ in range(runs):
        t0 = time.perf_counter(); fn(x); times.append((time.perf_counter() - t0) * 1000)
    return float(np.median(times))

x1 = x_laptop
torch.set_num_threads(os.cpu_count())
s32 = ort.InferenceSession(str(FP32), providers=["CPUExecutionProvider"])
s8 = ort.InferenceSession(str(INT8), providers=["CPUExecutionProvider"])
with torch.no_grad():
    lat = {
        "pytorch_fp32": bench(lambda x: wrapped(torch.from_numpy(x)), x1),
        "onnx_fp32": bench(lambda x: s32.run(None, {"image": x}), x1),
        "onnx_int8": bench(lambda x: s8.run(None, {"image": x}), x1),
    }
sizes = {"pytorch_fp32": (DRIVE_ROOT / "checkpoints" / NAME / "best.pt").stat().st_size / 1e6,
         "onnx_fp32": FP32.stat().st_size / 1e6, "onnx_int8": INT8.stat().st_size / 1e6}
summary = acc_table.copy()
summary["latency_ms"] = pd.Series(lat).round(1)
summary["fps"] = (1000 / summary["latency_ms"]).round(0)
summary["size_MB"] = pd.Series(sizes).round(1)
print(f"Colab CPU: {os.cpu_count()} cores — your laptop will be measured separately")
summary

## 8 · Choose the deployment model and save metadata

In [ ]:
drop = summary.loc["onnx_fp32", "accuracy"] - summary.loc["onnx_int8", "accuracy"]
DEPLOY = "distraction_int8.onnx" if drop <= 0.01 else "distraction_fp32.onnx"
print(f"INT8 accuracy change: {-drop:+.2%}  ->  deploy {DEPLOY}")

meta = {
    "model": NAME, "temperature": T, "deploy_file": DEPLOY,
    "image_size": IMG, "class_codes": CLASS_CODES, "class_names": CLASS_NAMES,
    "input": "RGB image, resized to image_size x image_size (bilinear), /255, ImageNet mean/std, NCHW",
    "output": "calibrated class probabilities (softmax(logits / temperature))",
    "test_metrics": summary.reset_index().rename(columns={"index": "variant"}).to_dict(orient="records"),
}
(EXPORTS / "model_meta.json").write_text(json.dumps(meta, indent=2))
summary.to_csv(DRIVE_ROOT / "results" / "export_benchmark.csv")
!ls -lh {EXPORTS}

---
✅ **Done.** Now download these files from Google Drive → `driver-monitoring/exports/`:
- `distraction_int8.onnx`
- `distraction_fp32.onnx`
- `model_meta.json`

and put them in your laptop project's **`models/`** folder (`C:\Projects\driver-monitoring-system\models\`).
`models/` is in `.gitignore`, so they won't be pushed to GitHub.